# 07 地震 · 中国及周边三维震源交互图

本笔记本按已确认方案，用本地主样本生成**独立离线 HTML**：经纬度投影到公里平面，深度向地下延伸，可旋转、筛选和开关地表图层。

只做分析输出，**不改写**主样本、精细清洗结果、既有平面图或分板块立体图，也不混入 USGS 或中国官方目录。HTML 内嵌数据与脚本，可用 `file://` 离线打开。

同目录的 `build_china_region_hypocenter_3d.py` 提供本本逐步调用的函数；也可在命令行运行该脚本重建页面。


## STEP 1 · 路径

METHOD: 用 `pathlib` 以本笔记本所在目录为起点，按项目相对布局查找 `07_analysis/07_earthquake_analysis_sample.csv`、PB2002、Natural Earth 与中国地块/断层。生成器内部还兼容当前工作目录，以及既有笔记本使用过的桌面项目文件夹。输出写到本目录，不覆盖旧图文件名。


In [ ]:
from pathlib import Path
import sys

HERE_CANDIDATES = [
    Path.cwd(),
    Path.cwd() / "interactive",
    Path.cwd() / "07_analysis" / "interactive",
]
HERE = None
for cand in HERE_CANDIDATES:
    if (cand / "build_china_region_hypocenter_3d.py").exists():
        HERE = cand.resolve()
        break
if HERE is None:
    raise FileNotFoundError(
        "找不到 build_china_region_hypocenter_3d.py。请在 interactive 目录中打开本笔记本，或先把工作目录设到项目根、07_analysis 或 interactive。"
    )

if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

import build_china_region_hypocenter_3d as H

paths = H.find_paths(HERE)
print("本目录:", HERE)
print("主样本:", paths["sample"])
print("分析目录:", paths["analysis_dir"])
print("板块多边形:", paths["plate_shp"] or paths["plate_json"])
print("板块边界:", paths["plate_bnd_json"] or paths["plate_bnd_shp"])
print("陆地:", paths["land_shp"])
print("国界:", paths["country_shp"])
print("地块目录:", paths["blocks_dir"])
print("断层:", paths["faults_gmt"])
print("输出目录:", paths["out_dir"])


## STEP 2 · 读取主样本

METHOD: 只读本地主样本 CSV。`Date_parsed` 按 UTC、ISO8601 解析；布尔标记按字符串 `true/1/yes` 兼容转换。不删除源文件中的行，也不改写该 CSV。随后各步的筛选只作用于本图所用的内存副本。


In [ ]:
df, global_stats = H.load_sample(paths["sample"])
for k, v in global_stats.items():
    print(f"{k}: {v}")
print("列名:", list(df.columns))


## STEP 3 · 区域筛选与深度完整性

METHOD: 按矩形 70–145°E、10–60°N 用坐标筛选，**不按** `Country` 或 `in_china`。时间为目录既有的 1965–2016 年覆盖。缺失深度的事件排除并报告条数；`flag_possible_duplicate` 为真的记录予以保留。深度分级：浅源 < 70 km，中源 70–300 km（含 300），深源 > 300 km。悬停日期只用 `Date_parsed` 的日期部分，时间使用目录 `Time` 字段原文，不把二者拼成未经证实的精确发震时刻。


In [ ]:
region, stats = H.filter_region(df, global_stats)
print(
    f"全球主样本 {stats['n_main_sample']} 条；坐标有效 {stats['n_valid_coords_global']}；"
    f"区域外 {stats['n_outside_region']}；区域内 {stats['n_region_coord']}。"
)
print(
    f"区域缺失深度排除 {stats['n_region_missing_depth_excluded']}（全球缺失深度 {stats['n_global_missing_depth']}）；"
    f"作图 {stats['n_region_used']}。"
)
print(
    f"浅/中/深 = {stats['n_shallow']}/{stats['n_intermediate']}/{stats['n_deep']}；"
    f"区域疑似重复保留 {stats['n_region_possible_duplicate_kept']}。"
)
print(
    f"震级 {stats['mag_min']}–{stats['mag_max']}，深度 {stats['depth_min']}–{stats['depth_max']} km，"
    f"年份 {stats['year_min']}–{stats['year_max']}。"
)
print("区域内 in_china 标记条数（仅供对照，不作筛选）:", stats["n_in_china_flag_in_region"])


## STEP 4 · PB2002 板块归属

METHOD: 复用既有分板块立体图的算法。读取 PB2002 板块多边形（优先 shapefile，失败则 GeoJSON），`make_valid` 后按 `PlateName` 合并被日界线切开的同一板块。点在多边形内（`within`）则归该板块，重叠时保留第一个；落在边界或缝隙则归最近板块。最近距离在经纬度平面上计算，与当时 `sjoin_nearest`（EPSG:4326）一致，不是大地线距离。先对全部坐标有效的主样本归属，并与既有 `07_st_3d_plate_counts.csv` 对照；再把结果连接到区域子集。不把中国活动地块边界当作板块或面域。


In [ ]:
plates = H.load_plates(paths)
spatial = df.loc[
    ~df.get("flag_coord_invalid", False) & df["Latitude"].notna() & df["Longitude"].notna()
].copy()
assigned_all, assign_all_stats = H.assign_plates(spatial, plates)
table_all = H.plate_count_table(assigned_all)
cmp = H.compare_plate_counts(table_all, paths.get("counts_csv"))
print(assign_all_stats["assign_rule"])
print("与既有计数表对照:", cmp)

region = region.merge(
    assigned_all[["PlateName", "PlateCode", "assign_rule", "plate_hover"]],
    left_index=True,
    right_index=True,
    how="left",
)
print(
    "区域归属 within/nearest/未归属:",
    int((region["assign_rule"] == "within").sum()),
    int((region["assign_rule"] == "nearest").sum()),
    int(region["PlateName"].isna().sum()),
)
print("区域板块前十：")
print(region["plate_hover"].value_counts().head(10).to_string())


## STEP 5 · 水平投影与深度轴

METHOD: 采用球面方位等距投影（Azimuthal Equidistant），地球半径 6371 km，中心 107.5°E、35°N，输出公里。这与既有立体图的 `+proj=aeqd +R=6371000` 一致。自中心出发的球面距离保持真实；远离中心时面积与方位有畸变。深度向下：z = -depth，地表为 0。随后在生成页面时用固定轴范围设置与跨度成正比的 aspectratio，默认 1:1，并可切换 5 倍垂直夸大（只改显示比例，不改 z 值）。


In [ ]:
import math

# 投影检验：中心应为原点；向北 1° 应等于 R × 1° 的弧长
x0, y0 = H.project_lonlat([H.PROJ_LON0], [H.PROJ_LAT0])
x1, y1 = H.project_lonlat([H.PROJ_LON0], [H.PROJ_LAT0 + 1])
arc = H.R_KM * math.radians(1)
print(f"投影中心 (x, y) = ({float(x0[0]):.6e}, {float(y0[0]):.6e})")
print(f"向北 1°: x={float(x1[0]):.6e}, y={float(y1[0]):.6f} km，理论弧长 {arc:.6f} km")

region = H.project_events(region)
print("z 与 -depth 一致:", (region["z_km"] + region["Depth"]).abs().max() < 1e-9)
print("x/y/z 范围 (km):")
print("  x", float(region["x_km"].min()), float(region["x_km"].max()))
print("  y", float(region["y_km"].min()), float(region["y_km"].max()))
print("  z", float(region["z_km"].min()), float(region["z_km"].max()))


## STEP 6 · 载入地表线并仅作显示抽稀

METHOD: 海岸线来自 Natural Earth 50m 陆地多边形的边界，国界来自 50m 国家多边形边界，板块边界来自 PB2002 边界线，中国一级/推断一级/二级活动地块与断层来自 GMT 线段。全部裁切到研究矩形，投影前对过长线段加密，投影后按公里容差抽稀，并放在 z = 0。抽稀只影响本图，不改原始文件。中国地块只画线，不填充为面。断层默认在页面中关闭。


In [ ]:
layers, layer_stats = H.load_all_layers(paths)
for key, st in layer_stats.items():
    print(
        f"{key:10s}  容差 {st['simplify_tol_km']} km  段 {st['n_segments_in']}→{st['n_segments_out']}  "
        f"顶点 {st['n_vertices_in']}→{st['n_vertices_out']}"
    )
    print("          源文件:", st["source"])


## STEP 7 · 写入独立离线 HTML

METHOD: 把事件坐标、悬停字段、图层折线、固定色标与轴范围、筛选范围组装为 JSON，与 Plotly 库、筛选脚本和页面模板一并写入同一个 HTML。不引用 CDN、在线地图、外部字体或密钥。若同名文件已存在，下面使用 `force=True` 以重建本目录中的标准成果文件；不会去覆盖其他旧图文件名。


In [ ]:
assign_region = {
    "n_region": int(len(region)),
    "n_within": int((region["assign_rule"] == "within").sum()),
    "n_nearest": int((region["assign_rule"] == "nearest").sum()),
    "n_unassigned": int(region["PlateName"].isna().sum()),
    "global": assign_all_stats,
}

payload = H.build_payload(region, layers, stats, layer_stats, assign_region)
out_html = HERE / "07_china_region_hypocenter_3d.html"
out_counts = HERE / "07_china_region_hypocenter_3d_counts.json"
H.write_html(payload, out_html, force=True)
H.write_counts_sidecar(payload, out_counts, force=True)

aspect = payload["scene"]["aspect_1"]
print("写入:", out_html, "bytes=", out_html.stat().st_size)
print("计数:", out_counts)
print("作图事件:", payload["meta"]["n_plotted"])
print("轴跨度 km:", payload["scene"]["xspan_km"], payload["scene"]["yspan_km"], payload["scene"]["zspan_km"])
print("1:1 aspectratio:", aspect)
print("生成时间:", payload["meta"]["built_at"])


## STEP 8 · 检查输出，确认旧文件未被覆盖

METHOD: 确认新 HTML 与计数文件已写入本目录；主样本、既有时空图和立体图笔记本、平面图文件仍在且本本没有向这些旧文件名写入。列出尚未用浏览器验证的项目。


In [ ]:
from datetime import datetime, timezone, timedelta

CST = timezone(timedelta(hours=8))

def cst_mtime(p):
    p = Path(p)
    if not p.exists():
        return "不存在"
    return datetime.fromtimestamp(p.stat().st_mtime, CST).strftime("%Y-%m-%d %H:%M:%S") + "（北京时间，Asia/Shanghai，UTC+8）"

print("新 HTML:", out_html.exists(), "bytes", out_html.stat().st_size if out_html.exists() else 0)
print("计数 JSON:", out_counts.exists())

protected = [
    paths["sample"],
    paths["analysis_dir"] / "07_earthquake_analysis_sample.ipynb",
    paths["analysis_dir"] / "07_earthquake_analysis_spatiotemporal.ipynb",
    paths["analysis_dir"] / "07_earthquake_analysis_spatiotemporal_3d.ipynb",
    paths["analysis_dir"] / "07_st_3d_plate_counts.csv",
    paths["analysis_dir"] / "figures" / "07_st_annual_mag.png",
    paths["analysis_dir"] / "figures" / "07_st_decade_mag.png",
    paths["analysis_dir"] / "figures" / "07_st_global_density_m7.png",
    paths["analysis_dir"] / "figures" / "07_st_global_depth.png",
]
print("既有文件仍在（本本未向这些路径写入）:")
for p in protected:
    pp = Path(p)
    print(" ", pp.name, "exists=", pp.exists(), "mtime=", cst_mtime(pp) if pp.exists() else "-")

print("待浏览器验证（本本未做）:")
print("- file:// 离线打开时网络面板应无外部请求")
print("- 旋转、缩放、俯视 / 侧视 / 复位")
print("- 1:1 与 5 倍夸大的视觉比例，以及筛选后比例是否保持")
print("- 年份、震级、深度分级组合筛选与空结果")
print("- 图层开关（断层默认关闭）且不遮挡地下点")
print("- 悬停字段与中文显示")
print("- 窗口缩放时控件不遮挡图像")
